In [ ]:
"""
Project 03: Customer Segmentation & CLV
Step 03: Evaluation Metrics & Pareto Analysis
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pickle
import os
import time
from tqdm import tqdm
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    mean_absolute_error,
    r2_score,
)

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 03: Evaluation Metrics & Pareto Analysis")
print("=" * 60)
start_time = time.time()

In [ ]:
# ── 1. Load Data & Models ──
proc_path = os.path.join(PROCESSED, "rfm_clv_data.csv")
df = pd.read_csv(proc_path, index_col=0)

In [ ]:
with open(os.path.join(MODELS, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)
with open(os.path.join(MODELS, "transformer.pkl"), "rb") as f:
    pt = pickle.load(f)

In [ ]:
print(f"Loaded results for {len(df)} customers.")

In [ ]:
# ── 2. Clustering Quality Metrics ──
print("\nComputing clustering metrics...")
features = ["Recency", "Frequency", "Monetary"]
X_transformed = pt.transform(df[features])
X_scaled = scaler.transform(X_transformed)

In [ ]:
sc = silhouette_score(X_scaled, df["Cluster"])
ch = calinski_harabasz_score(X_scaled, df["Cluster"])
db = davies_bouldin_score(X_scaled, df["Cluster"])

In [ ]:
print(f"  Silhouette Score:      {sc:.4f}")
print(f"  Calinski-Harabasz:     {ch:.2f}")
print(f"  Davies-Bouldin:        {db:.4f}")

In [ ]:
metrics_df = pd.DataFrame(
    {
        "Metric": ["Silhouette Score", "Calinski-Harabasz", "Davies-Bouldin"],
        "Value": [sc, ch, db],
    }
)
metrics_df.to_csv(os.path.join(MODELS, "performance_metrics.csv"), index=False)
print("Saved: performance_metrics.csv")

In [ ]:
# ── 3. CLV Model Performance ──
print("\nEvaluating CLV regression models...")
X_reg = np.column_stack([X_scaled, df["Cluster"]])
y_reg = df["CLV_Target"]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [ ]:
model_names = ["ridge", "rf", "gb"]
clv_results = []

In [ ]:
for name in tqdm(model_names, desc="Evaluating models"):
    model_path = os.path.join(MODELS, f"{name}_clv_model.pkl")
    if os.path.exists(model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
        clv_results.append({"Model": name.upper(), "MAE": mae, "RMSE": rmse, "R2": r2})
        print(
            f"  {name.upper():10s} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}"
        )

In [ ]:
clv_results_df = pd.DataFrame(clv_results)

In [ ]:
# ── 4. Pareto Analysis (80/20 Rule) ──
print("\nRunning Pareto analysis...")
df_pareto = df[["Monetary"]].copy()
df_pareto = df_pareto.sort_values(by="Monetary", ascending=False)
df_pareto["CumulativeSpend"] = df_pareto["Monetary"].cumsum()
df_pareto["%Spend"] = 100 * df_pareto["CumulativeSpend"] / df_pareto["Monetary"].sum()
df_pareto["%Customers"] = 100 * (np.arange(len(df_pareto)) + 1) / len(df_pareto)

In [ ]:
cutoff_80 = df_pareto[df_pareto["%Spend"] >= 80].iloc[0]
pareto_pct = cutoff_80["%Customers"]
print(f"PARETO FINDING: {pareto_pct:.1f}% of customers drive 80% of total revenue.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_pareto["%Customers"], df_pareto["%Spend"], color="#3b82f6", linewidth=2.5)
ax.fill_between(
    df_pareto["%Customers"], df_pareto["%Spend"], alpha=0.15, color="#3b82f6"
)
ax.axvline(
    x=pareto_pct,
    color="#ef4444",
    linestyle="--",
    linewidth=2,
    label=f"{pareto_pct:.1f}% of Customers",
)
ax.axhline(y=80, color="#ef4444", linestyle="--", linewidth=2, label="80% of Revenue")
ax.scatter([pareto_pct], [80], color="#ef4444", s=100, zorder=5)
ax.set_title(
    f"Pareto Analysis: {pareto_pct:.1f}% Drive 80% Revenue",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("% of Customers", fontsize=12)
ax.set_ylabel("% of Revenue", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "pareto_analysis.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: pareto_analysis.png")

In [ ]:
# ── 5. 3D Cluster Visualization ──
print("\nGenerating 3D cluster visualization...")
cluster_names = {0: "Champions", 1: "At-Risk", 2: "Loyal", 3: "About to Sleep"}
colors_map = {0: "#3b82f6", 1: "#ef4444", 2: "#10b981", 3: "#f59e0b"}

In [ ]:
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")

In [ ]:
for c in tqdm(range(4), desc="Plotting 3D clusters"):
    mask = df["Cluster"] == c
    ax.scatter(
        df.loc[mask, "Recency"],
        df.loc[mask, "Frequency"],
        df.loc[mask, "Monetary"],
        c=colors_map[c],
        label=cluster_names.get(c, f"Cluster {c}"),
        marker="o",
        s=40,
        alpha=0.6,
    )

In [ ]:
ax.set_xlabel("Recency (Days)", fontsize=11)
ax.set_ylabel("Frequency (Invoices)", fontsize=11)
ax.set_zlabel("Monetary (Spend)", fontsize=11)
ax.set_title("3D Customer Segmentation Matrix", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "rfm_3d_scatter.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: rfm_3d_scatter.png")

In [ ]:
# ── 6. CLV Prediction vs Actual ──
print("\nGenerating CLV prediction charts...")
best_model_name = clv_results_df.loc[clv_results_df["R2"].idxmax(), "Model"].lower()
with open(os.path.join(MODELS, f"{best_model_name}_clv_model.pkl"), "rb") as f:
    best_model = pickle.load(f)
y_pred_best = best_model.predict(X_test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

In [ ]:
# Scatter: Predicted vs Actual
axes[0].scatter(
    y_test,
    y_pred_best,
    alpha=0.4,
    s=20,
    color="#3b82f6",
    edgecolors="white",
    linewidth=0.5,
)
max_val = max(y_test.max(), y_pred_best.max())
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=2, label="Perfect Prediction")
axes[0].set_title(
    f"CLV: Predicted vs Actual ({best_model_name.upper()})",
    fontsize=14,
    fontweight="bold",
)
axes[0].set_xlabel("Actual CLV")
axes[0].set_ylabel("Predicted CLV")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

In [ ]:
# Residuals
residuals = y_test - y_pred_best
axes[1].scatter(
    y_pred_best,
    residuals,
    alpha=0.4,
    s=20,
    color="#10b981",
    edgecolors="white",
    linewidth=0.5,
)
axes[1].axhline(y=0, color="red", linestyle="--", linewidth=2)
axes[1].set_title("Residual Plot", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Predicted CLV")
axes[1].set_ylabel("Residual")
axes[1].grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "clv_predictions.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: clv_predictions.png")

In [ ]:
# ── 7. Cluster CLV Boxplot ──
fig, ax = plt.subplots(figsize=(10, 6))
cluster_data = [df[df["Cluster"] == c]["CLV_Target"].values for c in range(4)]
bp = ax.boxplot(
    cluster_data,
    labels=[cluster_names.get(i, f"Cluster {i}") for i in range(4)],
    patch_artist=True,
    showfliers=True,
)
for patch, color in zip(bp["boxes"], ["#3b82f6", "#ef4444", "#10b981", "#f59e0b"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title("CLV Distribution by Customer Segment", fontsize=14, fontweight="bold")
ax.set_ylabel("CLV Target ($)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "cluster_clv_boxplot.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print("Saved: cluster_clv_boxplot.png")

In [ ]:
# ── 8. Final Summary ──
print("\n" + "=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"  Total Customers:        {len(df)}")
print(f"  Clusters:               4 (K-Means)")
print(f"  Silhouette Score:       {sc:.4f}")
print(f"  Calinski-Harabasz:      {ch:.2f}")
print(f"  Davies-Bouldin:         {db:.4f}")
print(
    f"  VIP Outliers (DBSCAN):  {len(df[df.get('DBSCAN_Outlier', pd.Series()) == -1]) if 'DBSCAN_Outlier' in df.columns else 'N/A'}"
)
print(f"  Pareto:                 {pareto_pct:.1f}% customers = 80% revenue")
print(
    f"  Best CLV Model:         {best_model_name.upper()} (R2={clv_results_df['R2'].max():.4f})"
)

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 03 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")